In [0]:
import dlt
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

# ---- Paths for manifests (from your ingest notebook) ----
MANIFESTS_PATH = "abfss://lakehouse@buisdatastore.dfs.core.windows.net/bronze/fred_raw/manifests/"

# ---- Schema to parse observations if it's stored as STRING ----
obs_schema = ArrayType(StructType([
    StructField("realtime_start", StringType(), True),
    StructField("realtime_end",   StringType(), True),
    StructField("date",           StringType(), True),
    StructField("value",          StringType(), True),
]))

def _base():
    b = dlt.read_stream("canada_business.bronze.bronze_fred_raw_json")

    # If observations is already an array, keep it. If it's a string, parse it.
    observations_arr = F.when(
        F.col("observations").cast("string").startswith("["),
        F.from_json(F.col("observations").cast("string"), obs_schema)
    ).otherwise(F.lit(None).cast(obs_schema))

    return (
        b.withColumn("observations_arr", observations_arr)
         .withColumn("obs", F.explode_outer(F.col("observations_arr")))
         .withColumn("observation_date", F.to_date(F.col("obs.date")))
         .withColumn("value_raw", F.col("obs.value"))
         .withColumn(
             "value_double",
             F.when(F.col("value_raw").isin(".", "NA", ""), F.lit(None).cast("double"))
              .otherwise(F.col("value_raw").cast("double"))
         )
         .withColumn("obs_realtime_start", F.to_date(F.col("obs.realtime_start")))
         .withColumn("obs_realtime_end",   F.to_date(F.col("obs.realtime_end")))
         .withColumn("realtime_start",     F.to_date(F.col("realtime_start")))
         .withColumn("realtime_end",       F.to_date(F.col("realtime_end")))
    )

# ---------------- Silver: quarantine ----------------
@dlt.table(
  name="silver_fred_quarantine",
  comment="Rows that failed parsing or have missing/invalid fields"
)
def silver_fred_quarantine():
    x = _base()
    return (
        x.select(
            "series_id","_ingest_ts","_source_file",
            "observation_date","value_raw","value_double",
            "obs_realtime_start","obs_realtime_end","realtime_start","realtime_end",
            F.when(F.col("observations_arr").isNull(), F.lit("observations_parse_failed"))
             .when(F.col("obs").isNull(), F.lit("missing_observation_object"))
             .when(F.col("observation_date").isNull(), F.lit("invalid_or_missing_date"))
             .otherwise(F.lit("other")).alias("reject_reason")
        )
        .where(
            F.col("observations_arr").isNull() |
            F.col("obs").isNull() |
            F.col("observation_date").isNull()
        )
    )

# ---------------- Silver: clean observations ----------------
@dlt.table(
  name="silver_fred_observations",
  comment="Clean FRED observations (one row per series_id per date per ingest file)"
)
@dlt.expect_or_drop("valid_series", "series_id IS NOT NULL AND series_id <> ''")
@dlt.expect_or_drop("valid_date",   "observation_date IS NOT NULL")
def silver_fred_observations():
    x = _base()
    return x.select(
        "series_id","observation_date",
        "value_raw","value_double",
        "obs_realtime_start","obs_realtime_end",
        "realtime_start","realtime_end",
        "_ingest_ts","_source_file"
    )

# ---------------- Silver: basic metrics ----------------
@dlt.table(
  name="silver_fred_metrics",
  comment="Basic monitoring metrics"
)
def silver_fred_metrics():
    x = _base()
    return (
        x.groupBy()
         .agg(
             F.count("*").alias("rows_seen"),
             F.sum(F.when(F.col("observations_arr").isNull(), 1).otherwise(0)).alias("observations_parse_failed"),
             F.sum(F.when(F.col("observation_date").isNull(), 1).otherwise(0)).alias("invalid_date"),
             F.max(F.col("_ingest_ts")).alias("latest_ingest_ts")
         )
    )

# ---------------- Silver: ingest run audit (from manifests) ----------------
@dlt.table(
  name="silver_fred_runs",
  comment="One row per ingest run (from run_manifest_*.json)"
)
def silver_fred_runs():
    df = (
        spark.read
          .format("json")
          .option("multiLine", "true")
          .load(MANIFESTS_PATH)
    )

    per_series = df.select(
        "run_ts","catalog","schema","bronze_base","observation_start",
        F.explode_outer("series").alias("s")
    ).select(
        "run_ts","catalog","schema","bronze_base","observation_start",
        F.col("s.series_id").alias("series_id"),
        F.col("s.rows").cast("long").alias("rows"),
        F.col("s.raw_file").alias("raw_file")
    )

    return (
        per_series.groupBy("run_ts","catalog","schema","bronze_base","observation_start")
          .agg(
              F.countDistinct("series_id").alias("series_count"),
              F.countDistinct("raw_file").alias("file_count"),
              F.sum("rows").alias("total_rows")
          )
    )